# Kivy 应用 APK 打包工具（Google Colab 版）

本 Notebook 用于在 Google Colab 环境中将 Kivy 应用打包为 Android APK 文件。

## 使用说明

1. 依次运行下方的代码单元格
2. 首次构建需要 20~40 分钟，请耐心等待
3. 构建完成后可下载 APK 文件到本地或保存到 Google Drive

## 注意事项

- Colab 会话有时长限制（约 12 小时），请在会话有效期内完成构建
- 首次构建会下载 Android SDK/NDK，耗时较长
- 请确保您的项目包含 `buildozer.spec` 配置文件和 `main.py` 入口文件
- 如果构建失败，请查看末尾的「常见问题与解决方案」

---

## 第一步：环境准备

安装 Buildozer 及相关依赖，配置 Java 和 Android 构建环境。

### 1.1 检查系统环境

查看当前 Python 版本和系统信息。

In [ ]:
import sys
import platform

print("=" * 50)
print("系统信息检查")
print("=" * 50)
print(f"操作系统: {platform.system()} {platform.release()}")
print(f"Python 版本: {sys.version}")
print(f"Python 路径: {sys.executable}")
print(f"架构: {platform.machine()}")
print("=" * 50)
print("环境检查完成！")

### 1.2 安装系统依赖

安装 Buildozer 所需的系统依赖包（build-essential、git、zip、Java 等）。

In [ ]:
# 安装系统依赖
!apt-get update -qq
!apt-get install -y -qq \
    build-essential \
    git \
    zip \
    unzip \
    openjdk-17-jdk \
    openjdk-17-jre-headless \
    autoconf \
    libtool \
    pkg-config \
    libncurses5-dev \
    libncursesw5-dev \
    libtinfo5 \
    cmake \
    libffi-dev \
    libssl-dev \
    automake \
    libltdl-dev 2>&1 | tail -20

print("系统依赖安装完成！")

### 1.3 配置 Java 环境

设置 JAVA_HOME 环境变量。

In [ ]:
import os

# 设置 JAVA_HOME
java_home = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = f"{java_home}/bin:{os.environ['PATH']}"

# 验证 Java 安装
!java -version
!echo "JAVA_HOME=$JAVA_HOME"
print("Java 环境配置完成！")

### 1.4 安装 Buildozer 和 Cython

通过 pip 安装 Buildozer 构建工具和 Cython。

In [ ]:
# 升级 pip 并安装 buildozer 和 cython
!pip install --upgrade pip -q
!pip install buildozer cython==0.29.36 -q

# 验证安装
!buildozer --version
!cython --version
print("Buildozer 安装完成！")

### 1.5 配置 Android SDK/NDK 环境变量

设置 Android SDK 和 NDK 相关环境变量（Buildozer 会自动下载 SDK/NDK）。

In [ ]:
import os

# 设置 Android 相关环境变量
android_home = os.path.expanduser("~/.buildozer/android/platform/android-sdk")
ndk_home = os.path.expanduser("~/.buildozer/android/platform/android-ndk-r25b")

os.environ["ANDROID_HOME"] = android_home
os.environ["ANDROID_SDK_ROOT"] = android_home
os.environ["ANDROID_NDK_HOME"] = ndk_home

# 接受 SDK 许可协议（Buildozer 会自动处理，这里预先设置）
os.environ["ACCEPT_EULA"] = "1"

print("Android 环境变量已配置")
print(f"ANDROID_HOME: {android_home}")
print(f"ANDROID_NDK_HOME: {ndk_home}")
print("提示：SDK/NDK 将在首次构建时自动下载")

---

## 第二步：上传项目文件

请选择**一种方式**上传您的 Kivy 项目文件。

- **方式 A**：从本地上传 zip 压缩包
- **方式 B**：从 GitHub 克隆仓库

### 方式 A：上传 zip 压缩包

运行下方单元格，点击「选择文件」上传您的项目 zip 包。

**zip 包要求：**
- 根目录下必须包含 `buildozer.spec` 和 `main.py`
- 所有资源文件（图片、字体、配置文件等）都要包含在内
- 不要包含 `.buildozer`、`bin` 等构建目录

In [ ]:
import os
import zipfile
from google.colab import files

# 设置工作目录
WORK_DIR = "/content/myapp"
os.makedirs(WORK_DIR, exist_ok=True)

# 上传文件
print("请选择项目 zip 压缩包...")
uploaded = files.upload()

if uploaded:
    # 获取上传的文件名
    zip_filename = list(uploaded.keys())[0]
    print(f"\n已上传: {zip_filename}")
    
    # 解压到工作目录
    zip_path = os.path.join("/content", zip_filename)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(WORK_DIR)
    
    print(f"已解压到: {WORK_DIR}")
    
    # 检查目录结构
    print("\n项目文件列表:")
    for root, dirs, files_list in os.walk(WORK_DIR):
        level = root.replace(WORK_DIR, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        if level < 3:  # 只显示前3层
            subindent = ' ' * 2 * (level + 1)
            for file in files_list[:10]:  # 每个目录最多显示10个文件
                print(f"{subindent}{file}")
            if len(files_list) > 10:
                print(f"{subindent}... 还有 {len(files_list) - 10} 个文件")
else:
    print("未上传文件，请重新运行此单元格")

### 方式 B：从 GitHub 克隆

如果您的项目托管在 GitHub 上，可以直接克隆。

修改下方的仓库地址，然后运行单元格。

In [ ]:
import os

# ========== 请修改以下配置 ==========
GITHUB_REPO_URL = "https://github.com/yourusername/your-kivy-app.git"  # GitHub 仓库地址
GIT_BRANCH = "main"  # 分支名称，默认 main
# ====================================

# 设置工作目录
WORK_DIR = "/content/myapp"

# 克隆仓库
print(f"正在克隆仓库: {GITHUB_REPO_URL}")
print(f"分支: {GIT_BRANCH}")

!rm -rf {WORK_DIR}
!git clone --branch {GIT_BRANCH} --depth 1 {GITHUB_REPO_URL} {WORK_DIR}

# 检查克隆结果
if os.path.exists(WORK_DIR):
    print(f"\n克隆成功！项目目录: {WORK_DIR}")
    print("\n项目文件列表:")
    for root, dirs, files_list in os.walk(WORK_DIR):
        level = root.replace(WORK_DIR, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        if level < 3:
            subindent = ' ' * 2 * (level + 1)
            for file in files_list[:10]:
                print(f"{subindent}{file}")
            if len(files_list) > 10:
                print(f"{subindent}... 还有 {len(files_list) - 10} 个文件")
else:
    print("克隆失败，请检查仓库地址是否正确")

### 2.3 验证项目结构

上传/克隆完成后，运行下方单元格检查项目是否包含必要文件。

In [ ]:
import os

WORK_DIR = "/content/myapp"

print("=" * 50)
print("项目结构检查")
print("=" * 50)

# 检查工作目录是否存在
if not os.path.exists(WORK_DIR):
    print(f"❌ 工作目录不存在: {WORK_DIR}")
    print("请先运行「方式 A」或「方式 B」上传项目文件")
else:
    # 检查必要文件
    main_py = os.path.join(WORK_DIR, "main.py")
    buildozer_spec = os.path.join(WORK_DIR, "buildozer.spec")
    
    has_main = os.path.exists(main_py)
    has_spec = os.path.exists(buildozer_spec)
    
    if has_main:
        print(f"✅ main.py 存在")
    else:
        print(f"❌ main.py 不存在！")
    
    if has_spec:
        print(f"✅ buildozer.spec 存在")
    else:
        print(f"⚠️  buildozer.spec 不存在，将在第三步生成默认配置")
    
    # 列出项目根目录文件
    print(f"\n项目根目录内容:")
    for item in sorted(os.listdir(WORK_DIR)):
        item_path = os.path.join(WORK_DIR, item)
        if os.path.isdir(item_path):
            print(f"  📁 {item}/")
        else:
            size = os.path.getsize(item_path)
            print(f"  📄 {item} ({size:,} bytes)")
    
    print("\n" + "=" * 50)
    if has_main:
        print("项目检查通过，可以继续下一步！")
    else:
        print("项目检查不通过，请检查项目文件！")

---

## 第三步：配置项目

检查并修改 buildozer.spec 配置文件。

### 3.1 检查 / 生成 buildozer.spec

如果项目中没有 buildozer.spec，将自动生成默认配置文件。

In [ ]:
import os

WORK_DIR = "/content/myapp"
spec_path = os.path.join(WORK_DIR, "buildozer.spec")

%cd {WORK_DIR}

if os.path.exists(spec_path):
    print("找到 buildozer.spec 配置文件")
    print("-" * 50)
    with open(spec_path, 'r', encoding='utf-8') as f:
        content = f.read()
        print(content[:2000])  # 显示前2000字符
        if len(content) > 2000:
            print(f"\n... (文件共 {len(content)} 字符，已截断显示)")
else:
    print("未找到 buildozer.spec，正在生成默认配置...")
    !buildozer init
    print("默认配置文件已生成")
    print("\n请根据您的项目修改配置后再继续构建")

### 3.2 修改应用配置（可选）

如果需要修改应用名称、包名、版本号等信息，可以运行下方单元格。
修改完成后会自动更新 buildozer.spec 文件。

In [ ]:
import os

WORK_DIR = "/content/myapp"
spec_path = os.path.join(WORK_DIR, "buildozer.spec")

# ========== 配置项 ==========
# 应用标题（显示在桌面图标下方）
APP_TITLE = "My Kivy App"

# 包名（反向域名格式，如 org.example.myapp）
PACKAGE_DOMAIN = "org.example"
PACKAGE_NAME = "myapp"

# 版本号
APP_VERSION = "1.0"

# 屏幕方向: landscape（横屏）, portrait（竖屏）, sensor（自适应）
ORIENTATION = "landscape"

# 完整入口文件名（必须是项目中的 Python 主文件）
ENTRY_POINT = "main.py"
# ============================

if not os.path.exists(spec_path):
    print("未找到 buildozer.spec，请先运行上一步！")
else:
    with open(spec_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # 使用正则替换配置项
    import re
    
    # 修改 title
    content = re.sub(r'^title\s*=.*$', f'title = {APP_TITLE}', content, flags=re.MULTILINE)
    
    # 修改 package.name
    content = re.sub(r'^package\.name\s*=.*$', f'package.name = {PACKAGE_NAME}', content, flags=re.MULTILINE)
    
    # 修改 package.domain
    content = re.sub(r'^package\.domain\s*=.*$', f'package.domain = {PACKAGE_DOMAIN}', content, flags=re.MULTILINE)
    
    # 修改 version
    content = re.sub(r'^version\s*=.*$', f'version = {APP_VERSION}', content, flags=re.MULTILINE)
    
    # 修改 orientation
    content = re.sub(r'^orientation\s*=.*$', f'orientation = {ORIENTATION}', content, flags=re.MULTILINE)
    
    # 写回文件
    with open(spec_path, 'w', encoding='utf-8') as f:
        f.write(content)
    
    print("✅ 配置已更新！")
    print(f"   应用名称: {APP_TITLE}")
    print(f"   包名: {PACKAGE_DOMAIN}.{PACKAGE_NAME}")
    print(f"   版本: {APP_VERSION}")
    print(f"   方向: {ORIENTATION}")

### 3.3 检查依赖配置（可选）

查看和修改项目的 Python 依赖（requirements）。

In [ ]:
import os
import re

WORK_DIR = "/content/myapp"
spec_path = os.path.join(WORK_DIR, "buildozer.spec")

if not os.path.exists(spec_path):
    print("未找到 buildozer.spec，请先运行前面的步骤！")
else:
    with open(spec_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # 提取 requirements
    match = re.search(r'^requirements\s*=\s*(.+)$', content, re.MULTILINE)
    if match:
        current_req = match.group(1).strip()
        print(f"当前依赖: {current_req}")
    else:
        current_req = "python3,kivy"
        print(f"未找到 requirements，使用默认值: {current_req}")
    
    print("\n提示：")
    print("  - 常用依赖: python3, kivy, pillow, numpy, requests")
    print("  - 如需修改依赖，请直接编辑 buildozer.spec 文件")
    print("  - 注意：不是所有 PyPI 包都能在 Android 上运行")

---

## 第四步：开始构建 APK

运行 Buildozer 构建 Android APK。

**⚠️ 重要提示：**
- 首次构建需要 **20~40 分钟**，请耐心等待
- 首次构建会自动下载 Android SDK 和 NDK（约 2GB）
- 构建过程中请勿关闭此页面
- 如遇构建失败，请查看末尾的「常见问题」

In [ ]:
import os
import time

WORK_DIR = "/content/myapp"
%cd {WORK_DIR}

print("=" * 60)
print("开始构建 APK")
print("=" * 60)
print(f"工作目录: {WORK_DIR}")
print(f"开始时间: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)
print("\n⚠️  首次构建可能需要 20-40 分钟")
print("⚠️  请保持页面打开，不要中断构建")
print("\n正在启动构建...\n")

# 运行 buildozer 构建 debug 版 APK
start_time = time.time()

!buildozer android debug 2>&1

end_time = time.time()
elapsed = end_time - start_time

print("\n" + "=" * 60)
print(f"构建完成！耗时: {int(elapsed // 60)} 分 {int(elapsed % 60)} 秒")
print("=" * 60)

# 检查构建结果
bin_dir = os.path.join(WORK_DIR, "bin")
if os.path.exists(bin_dir):
    apk_files = [f for f in os.listdir(bin_dir) if f.endswith('.apk')]
    if apk_files:
        print(f"\n✅ 构建成功！生成的 APK 文件:")
        for apk in apk_files:
            apk_path = os.path.join(bin_dir, apk)
            size_mb = os.path.getsize(apk_path) / (1024 * 1024)
            print(f"   - {apk} ({size_mb:.2f} MB)")
    else:
        print("\n❌ 未找到 APK 文件，构建可能失败了")
else:
    print("\n❌ bin 目录不存在，构建失败")

---

## 第五步：下载 APK

构建成功后，可以通过以下方式获取 APK 文件。

### 5.1 直接下载到本地

运行下方单元格，APK 文件将自动下载到您的电脑。

In [ ]:
import os
from google.colab import files

WORK_DIR = "/content/myapp"
bin_dir = os.path.join(WORK_DIR, "bin")

if os.path.exists(bin_dir):
    apk_files = [f for f in os.listdir(bin_dir) if f.endswith('.apk')]
    if apk_files:
        print(f"找到 {len(apk_files)} 个 APK 文件:")
        for i, apk in enumerate(apk_files, 1):
            apk_path = os.path.join(bin_dir, apk)
            size_mb = os.path.getsize(apk_path) / (1024 * 1024)
            print(f"  {i}. {apk} ({size_mb:.2f} MB)")
        
        print("\n正在下载 APK 文件...")
        for apk in apk_files:
            apk_path = os.path.join(bin_dir, apk)
            files.download(apk_path)
            print(f"✅ 已触发下载: {apk}")
    else:
        print("❌ bin 目录中没有找到 APK 文件")
        print("请先运行第四步构建 APK")
else:
    print("❌ bin 目录不存在，请先运行第四步构建 APK")

### 5.2 保存到 Google Drive（推荐）

将 APK 保存到您的 Google Drive，方便随时下载和分享。
运行下方单元格，按提示授权访问 Google Drive。

In [ ]:
import os
import shutil
from google.colab import drive

WORK_DIR = "/content/myapp"
bin_dir = os.path.join(WORK_DIR, "bin")

# 挂载 Google Drive
print("正在连接 Google Drive...")
drive.mount('/content/drive')

# 设置保存目录
drive_save_dir = "/content/drive/MyDrive/APK_Builds"
os.makedirs(drive_save_dir, exist_ok=True)

if os.path.exists(bin_dir):
    apk_files = [f for f in os.listdir(bin_dir) if f.endswith('.apk')]
    if apk_files:
        print(f"\n找到 {len(apk_files)} 个 APK 文件:")
        for apk in apk_files:
            src_path = os.path.join(bin_dir, apk)
            dst_path = os.path.join(drive_save_dir, apk)
            
            # 如果文件已存在，添加时间戳
            if os.path.exists(dst_path):
                import time
                name, ext = os.path.splitext(apk)
                timestamp = time.strftime("%Y%m%d_%H%M%S")
                dst_path = os.path.join(drive_save_dir, f"{name}_{timestamp}{ext}")
            
            shutil.copy2(src_path, dst_path)
            size_mb = os.path.getsize(dst_path) / (1024 * 1024)
            print(f"  ✅ {apk} -> {os.path.basename(dst_path)} ({size_mb:.2f} MB)")
        
        print(f"\nAPK 已保存到 Google Drive: {drive_save_dir}")
    else:
        print("❌ bin 目录中没有找到 APK 文件")
else:
    print("❌ bin 目录不存在，请先运行第四步构建 APK")

### 5.3 查看构建产物

列出 bin 目录下的所有文件。

In [ ]:
import os

WORK_DIR = "/content/myapp"
bin_dir = os.path.join(WORK_DIR, "bin")

if os.path.exists(bin_dir):
    print("构建产物目录:")
    print("-" * 50)
    for f in sorted(os.listdir(bin_dir)):
        fpath = os.path.join(bin_dir, f)
        size = os.path.getsize(fpath)
        if size > 1024 * 1024:
            size_str = f"{size / (1024*1024):.2f} MB"
        elif size > 1024:
            size_str = f"{size / 1024:.2f} KB"
        else:
            size_str = f"{size} B"
        print(f"  {f}  ({size_str})")
else:
    print("bin 目录不存在，请先构建 APK")

---

## 常见问题与解决方案

### Q1: 构建失败，提示 `Command failed: gradlew assembleDebug`

**可能原因：**
- 网络问题导致依赖下载失败
- 内存不足
- Java 版本不兼容

**解决方案：**
1. 重新运行构建命令（网络问题通常重试即可）
2. 确保使用的是 Java 17
3. 清理构建缓存后重试：`!buildozer android clean`

### Q2: 构建超时，Colab 会话断开

**可能原因：** Colab 免费版有会话时长限制

**解决方案：**
1. 使用 Google Drive 保存中间产物（.buildozer 目录）
2. 下次构建时可以跳过 SDK/NDK 下载，节省时间
3. 考虑使用 Colab Pro 获得更长的会话时长

### Q3: APK 安装后闪退

**可能原因：**
- 缺少必要的权限
- 资源文件路径问题
- 使用了不兼容的 Python 库

**解决方案：**
1. 检查 buildozer.spec 中的 android.permissions 配置
2. 确保所有文件路径使用相对路径
3. 使用 `adb logcat` 查看崩溃日志

### Q4: 如何生成 release 版 APK？

**答：** 将构建命令改为：
```python
!buildozer android release
```
注意：release 版需要签名证书才能安装。

### Q5: 如何加速构建？

**技巧：**
1. 首次构建后，将 `.buildozer` 目录保存到 Google Drive
2. 后续构建前从 Drive 恢复，可节省 SDK/NDK 下载时间
3. 使用 Colab 的 GPU 或 TPU 运行时（加速有限）

### Q6: 支持哪些 Python 库？

**答：** 不是所有 PyPI 包都支持 Android。常见的支持库包括：
- kivy, kivymd
- pillow
- numpy
- requests
- sqlite3（内置）

需要原生编译的库（如 opencv、pandas）可能需要额外配置。

### Q7: 如何查看详细构建日志？

**答：** 运行以下命令查看详细日志：
```python
!buildozer -v android debug  # 详细输出
```

---

## 更多资源

- [Buildozer 官方文档](https://buildozer.readthedocs.io/)
- [Kivy 官方文档](https://kivy.org/doc/stable/)
- [Kivy Android 打包指南](https://kivy.org/doc/stable/guide/packaging-android.html)
- [Python for Android 文档](https://python-for-android.readthedocs.io/)